In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier  # RF用にインポート
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. 特徴量リストの定義
# ==========================================
function_features = [
    "機能語率", "ひらがな率", "句点頻度", "読点頻度", "助詞割合", "助動詞割合", "連体詞割合", "接続詞割合", "感動詞割合",
    "助詞_は", "助詞_に", "助詞_て", "助詞_と", "助詞_か", "助詞_が", "助詞_で", "助詞_や", "助詞_を", "助詞_の",
    "助詞_ほど", "助詞_まで", "助詞_も", "助詞_から", "助詞_たり", "助詞_など", "助詞_ながら", "助詞_だけ", "助詞_ば",
    "助詞_へ", "助詞_のみ", "助詞_しも", "助詞_しか", "助詞_すら", "助詞_より", "助詞_こそ", "助詞_つつ", "助詞_くらい",
    "助詞_かしら", "助詞_ずつ"
]

content_features = [
    "内容語率", "動詞割合", "名詞割合", "形容詞割合", "副詞割合", "形状詞割合", "漢字率", "カタカナ率", "英数字率"
]

# 全特徴量をベースとして定義
base_features = function_features + content_features

# ==========================================
# 2. データの読み込みと前処理
# ==========================================
current_dir = os.getcwd()
if not current_dir.endswith("AI確信度"):
    current_dir = os.path.join(current_dir, "AI確信度")

# 訓練データとテストデータをそれぞれのファイルから読み込み
doc_csv_path = os.path.join(current_dir, "train_Confidence-Level_one_sentence_stylometry_features.csv")
sent_csv_path = os.path.join(current_dir, "test_Confidence-Level_one_sentence_stylometry_features.csv")

df_doc = pd.read_csv(doc_csv_path).fillna(0)     # 訓練用ベース
df_sent = pd.read_csv(sent_csv_path).fillna(0)   # テスト用ベース

# ★ テストデータ等に存在しない特徴量列を0で自動追加
for col in base_features:
    if col not in df_doc.columns:
        df_doc[col] = 0.0
    if col not in df_sent.columns:
        df_sent[col] = 0.0

# doc_id の作成と文章の並び順番号の付与
df_doc["doc_id"] = df_doc["filename"].str.replace(r"_s\d+$", "", regex=True)
df_doc["sent_order"] = df_doc.groupby("doc_id").cumcount() + 1

df_sent["doc_id"] = df_sent["filename"].str.replace(r"_s\d+$", "", regex=True)
df_sent["sent_order"] = df_sent.groupby("doc_id").cumcount() + 1

# ==========================================
# 3. 文書単位での分割の廃止 (全件をそのまま割り当て)
# ==========================================
# ランダムサンプリングは行わず、ファイルをそのまま分離して使用します
train_docs = df_doc["doc_id"].unique()
test_docs = df_sent["doc_id"].unique()

# 訓練データ（文章単位）の抽出
df_train = df_doc.copy()

# ==========================================
# 4. 特徴量セットごとの学習とテストデータの予測（ランダムフォレスト版）
# ==========================================
feature_sets = {
    "All": base_features,
    "Function": [f for f in base_features if f in function_features],
    "Content": [f for f in base_features if f in content_features],
}

for set_name, features in feature_sets.items():
    print(
        f"【処理中】{set_name} 特徴量でモデル（Random Forest）を学習・予測中... (特徴量数: {len(features)})"
    )

    # 各特徴量セットごとに訓練データを準備
    X_train_set = df_train[features].values
    y_train_set = df_train["label"].values

    # 標準化
    scaler_set = StandardScaler()
    X_train_scaled = scaler_set.fit_transform(X_train_set)

    # ランダムフォレストモデルの定義と学習
    model_set = RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1
    )
    model_set.fit(X_train_scaled, y_train_set)

    # ★ 追加：特徴量重要度（寄与度）の抽出とCSV保存
    df_importance = pd.DataFrame(
        {"特徴量": features, "重要度": model_set.feature_importances_}
    ).sort_values(by="重要度", ascending=False).reset_index(drop=True)

    importance_filename = f"RF_feature_importance_{set_name}.csv"
    importance_path = os.path.join(current_dir, importance_filename)
    df_importance.to_csv(importance_path, index=False, encoding="utf-8-sig")
    print(f"-> 特徴量重要度を '{importance_filename}' に保存しました。")

    # テスト文書ごとにループ (テスト専用の df_sent から抽出)
    results = []
    for doc_id in test_docs:
        df_doc_sents = df_sent[df_sent["doc_id"] == doc_id].sort_values(
            "sent_order"
        )
        if df_doc_sents.empty:
            continue

        true_label = df_doc_sents["label"].iloc[0]
        proba_history = []

        for idx, row in df_doc_sents.iterrows():
            sent_order = row["sent_order"]

            # 1文ずつの特徴量を取得して標準化
            feat = row[features].values.reshape(1, -1)
            feat_scaled = scaler_set.transform(feat)

            # AI確率の予測
            ai_index = list(model_set.classes_).index("AI")
            pred_proba_single = model_set.predict_proba(feat_scaled)[0][
                ai_index
            ]

            # 予測確率を履歴に追加し、ここまでの平均確率を計算（累積ロジック）
            proba_history.append(pred_proba_single)
            cum_proba_avg = sum(proba_history) / len(proba_history)

            # 閾値0.5で暫定判定
            pred_label = "AI" if cum_proba_avg >= 0.5 else "Human"

            results.append(
                {
                    "doc_id": doc_id,
                    "sent_order": sent_order,
                    "label": true_label,
                    "AI確率": cum_proba_avg,
                    "予測ラベル": pred_label,
                }
            )

    # RF版の結果としてCSVに保存（頭に rf_ を付与）
    df_results = pd.DataFrame(results)
    output_filename = f"rf_predict_proba_split_{set_name}_results.csv"
    output_path = os.path.join(current_dir, output_filename)
    df_results.to_csv(output_path, index=False)
    print(f"-> 予測結果を '{output_filename}' に保存しました。\n")

print("【すべての処理が完了しました】")

【処理中】All 特徴量でモデル（Random Forest）を学習・予測中... (特徴量数: 48)
-> 特徴量重要度を 'RF_feature_importance_All.csv' に保存しました。
-> 予測結果を 'rf_predict_proba_split_All_results.csv' に保存しました。

【処理中】Function 特徴量でモデル（Random Forest）を学習・予測中... (特徴量数: 39)
-> 特徴量重要度を 'RF_feature_importance_Function.csv' に保存しました。
-> 予測結果を 'rf_predict_proba_split_Function_results.csv' に保存しました。

【処理中】Content 特徴量でモデル（Random Forest）を学習・予測中... (特徴量数: 9)
-> 特徴量重要度を 'RF_feature_importance_Content.csv' に保存しました。
-> 予測結果を 'rf_predict_proba_split_Content_results.csv' に保存しました。

【すべての処理が完了しました】


In [8]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# ==========================================
# 1. 特徴量リストの定義とモード選択
# ==========================================
function_features = [
    "機能語率", "ひらがな率", "句点頻度", "読点頻度", "助詞割合", "助動詞割合", "連体詞割合", "接続詞割合", "感動詞割合",
    "助詞_は", "助詞_に", "助詞_て", "助詞_と", "助詞_か", "助詞_が", "助詞_で", "助詞_や", "助詞_を", "助詞_の",
    "助詞_ほど", "助詞_まで", "助詞_も", "助詞_から", "助詞_たり", "助詞_など", "助詞_ながら", "助詞_だけ", "助詞_ば",
    "助詞_へ", "助詞_のみ", "助詞_しも", "助詞_しか", "助詞_すら", "助詞_より", "助詞_こそ", "助詞_つつ", "助詞_くらい",
    "助詞_かしら", "助詞_ずつ"
]

content_features = [
    "内容語率", "動詞割合", "名詞割合", "形容詞割合", "副詞割合", "形状詞割合", "漢字率", "カタカナ率", "英数字率"
]

# ★ 実験したい特徴量セットを選定 ("All" or "Function" or "Content")
feature_mode = "Content"

if feature_mode == "All":
    base_features = function_features + content_features
elif feature_mode == "Function":
    base_features = function_features
elif feature_mode == "Content":
    base_features = content_features

print(f"【実験設定】選択された特徴量モード: {feature_mode} (ベース特徴量数: {len(base_features)})")

# ==========================================
# 2. データの読み込みと前処理
# ==========================================
current_dir = os.getcwd()
if not current_dir.endswith("AI確信度"):
    current_dir = os.path.join(current_dir, "AI確信度")

doc_csv_path = os.path.join(current_dir, "train_Confidence-Level_one_sentence_stylometry_features.csv")
sent_csv_path = os.path.join(current_dir, "test_Confidence-Level_one_sentence_stylometry_features.csv")

df_doc = pd.read_csv(doc_csv_path).fillna(0)
df_sent = pd.read_csv(sent_csv_path).fillna(0)

# 不足している特徴量列を0で自動追加
for col in base_features:
    if col not in df_doc.columns:
        df_doc[col] = 0.0
    if col not in df_sent.columns:
        df_sent[col] = 0.0

# doc_id と sent_order の整理
df_doc["doc_id"] = df_doc["filename"].str.replace(r"_s\d+$", "", regex=True)
df_doc = df_doc.sort_values(["doc_id", "filename"]) # 順序を揃える
df_doc["sent_order"] = df_doc.groupby("doc_id").cumcount() + 1

df_sent["doc_id"] = df_sent["filename"].str.replace(r"_s\d+$", "", regex=True)
df_sent = df_sent.sort_values(["doc_id", "filename"]) # 順序を揃える
df_sent["sent_order"] = df_sent.groupby("doc_id").cumcount() + 1


# ==========================================
# 3. ★ 特徴量の累積平均化（学習・テスト共通）
# ==========================================
# 各ドキュメントごとに、1文目からその文(sent_order)までの特徴量の累積平均（expanding mean）を算出
print("【前処理】特徴量の累積平均（段階的増加）を計算中...")
df_doc[base_features] = df_doc.groupby("doc_id")[base_features].transform(lambda x: x.expanding().mean())
df_sent[base_features] = df_sent.groupby("doc_id")[base_features].transform(lambda x: x.expanding().mean())


# ==========================================
# 4. 訓練データ（段階的プロセスを含む全状態）の抽出と学習
# ==========================================
df_train = df_doc.copy()

X_train = df_train[base_features].values
y_train = df_train["label"].values

input_dim = X_train.shape[1]
print(f"【学習設定】訓練データ（総状態数）: {X_train.shape[0]} 件, 特徴量次元数: {input_dim}")

# 標準化と学習
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# ランダムフォレストモデルの学習（Notebookの設定に準拠）
model = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
model.fit(X_train_scaled, y_train)


# ==========================================
# 5. テストデータの予測と段階的確率の算出
# ==========================================
test_docs = df_sent["doc_id"].unique()
results = []

for doc_id in test_docs:
    df_doc_sents = df_sent[df_sent["doc_id"] == doc_id].sort_values("sent_order")
    if df_doc_sents.empty:
        continue
    true_label = df_doc_sents["label"].iloc[0]
    
    for idx, row in df_doc_sents.iterrows():
        sent_order = row["sent_order"]
        
        # 1. すでに累積平均化されている特徴量を取得して標準化
        feat = row[base_features].values.reshape(1, -1)
        feat_scaled = scaler.transform(feat)
        
        # 2. 累積特徴量に対するAI確率を予測
        ai_index = list(model.classes_).index("AI")
        pred_proba_cum = model.predict_proba(feat_scaled)[0][ai_index]
        
        # 3. 予測確率に基づいて、現時点での暫定ラベルを決定
        pred_label = "AI" if pred_proba_cum >= 0.5 else "Human"
        
        results.append({
            "doc_id": doc_id,
            "sent_order": sent_order,
            "label": true_label,
            "AI確率": pred_proba_cum,
            "予測ラベル": pred_label
        })

# ==========================================
# 6. CSV保存
# ==========================================
df_output = pd.DataFrame(results)
# ファイル名の競合を避けるため、先頭に `more_rf_` を付与して保存します
output_filename = f"more_rf_predict_proba_split_{feature_mode}_results.csv"
output_path = os.path.join(current_dir, output_filename)
df_output.to_csv(output_path, index=False)

print(f"\n[完了] RFを用いた、学習・テスト共に段階的累積特徴量を適用した予測結果が '{output_filename}' に保存されました！")
print(df_output.head(15))

【実験設定】選択された特徴量モード: Content (ベース特徴量数: 9)
【前処理】特徴量の累積平均（段階的増加）を計算中...
【学習設定】訓練データ（総状態数）: 721 件, 特徴量次元数: 9

[完了] RFを用いた、学習・テスト共に段階的累積特徴量を適用した予測結果が 'more_rf_predict_proba_split_Content_results.csv' に保存されました！
                      doc_id  sent_order label  AI確率  予測ラベル
0   test_ChatGPT_text_01.txt           1    AI  0.21  Human
1   test_ChatGPT_text_01.txt           2    AI  0.11  Human
2   test_ChatGPT_text_01.txt           3    AI  0.25  Human
3   test_ChatGPT_text_01.txt           4    AI  0.27  Human
4   test_ChatGPT_text_01.txt           5    AI  0.10  Human
5   test_ChatGPT_text_01.txt           6    AI  0.21  Human
6   test_ChatGPT_text_01.txt           7    AI  0.28  Human
7   test_ChatGPT_text_01.txt           8    AI  0.40  Human
8   test_ChatGPT_text_01.txt           9    AI  0.40  Human
9   test_ChatGPT_text_01.txt          10    AI  0.55     AI
10  test_ChatGPT_text_02.txt           1    AI  0.94     AI
11  test_ChatGPT_text_02.txt           2    AI  0.94     AI
12  test_ChatGPT